In [1]:
from functions import *

In [2]:
a_vec = [763, 679, 397, 61, 697, 373, 
         289, 257, 625, 41, 193, 449]
b_vec = [435, 69, 330, 18, 612, 246, 
         496, 640, 200, 524, 672, 672] 

In [ ]:
G = generate_g(a_vec)
all_indices = [i for i in range(l_h**2)]
gb_indices = [3, 8]
ga_indices = [i for i in all_indices if i not in gb_indices]
Ga = G.extract(ga_indices, list(range(G.cols)))
Gb = G.extract(gb_indices, list(range(G.cols)))
basis_a = solve_modular_kernel(Ga, P)
V = Matrix.hstack(*basis_a)
Gb_prime = (Gb * V).applyfunc(lambda x: x % P)

In [4]:
# a_vec_matrix = sp.Matrix(a_vec)
# b_vec_matrix = sp.Matrix(b_vec)

# resulta = Ga @ b_vec_matrix

# resultb = Gb @ b_vec_matrix
# resulta % P, resultb % P #これが非ゼロなことは確認済み

In [5]:
forbidden_bases = []
for k in range(Gb_prime.rows):
    row = Gb_prime.row(k)
    basis = solve_modular_kernel(row, P)
    forbidden_bases.append(basis)

In [6]:
cycles = generate_cycles(6)
h_x, h_z = generate_h_xz()
constraints = generate_constraints(cycles, a_vec, h_x, h_z)

In [7]:
for c in constraints:
    c_prime = (Matrix([c]) * V).applyfunc(lambda x: x % P)
    basis_k = solve_modular_kernel(c_prime, P)
    forbidden_bases.append(basis_k)

In [8]:
from sympy import Matrix

# 1. b_vec を行列形式に変換
b_mat = Matrix(b_vec)

# 2. V * x = b を有理数体（Q）上で解く
# V は main.ipynb で定義された Matrix.hstack(*basis_a)
try:
    # V が正則であれば x = V^-1 * b が求まる
    x_rational = V.solve(b_mat)

    # 3. 有理数の解 a/b を整数 a * inv(b, P) (mod P) に変換する関数
    def to_mod_p(val, p):
        num, den = val.as_numer_denom()
        # Python 3.8+ の pow(den, -1, p) はモジュラ逆数を計算する
        return (int(num) * pow(int(den), -1, p)) % p

    # 各要素に適用
    coefficients = x_rational.applyfunc(lambda v: to_mod_p(v, P))

    print("線形結合の係数ベクトル x:")
    display(coefficients)
    
    # 検算: V * x % P が b_vec と一致するか確認
    check = (V * coefficients).applyfunc(lambda x: x % P)
    if check == b_mat.applyfunc(lambda x: x % P):
        print("検算成功: 一致しました。")
    else:
        print("警告: 検算に失敗しました。")

except Exception as e:
    print(f"解を求めることができませんでした: {e}")

# coefficients が Gb の制約（禁止領域）に触れているか確認
gb_check = (Gb_prime * coefficients).applyfunc(lambda x: x % P)

print("Gb_prime * x (mod P) の結果:")
display(gb_check)

# 0 になっている要素があるか確認
forbidden_rows = [i for i, val in enumerate(gb_check) if val == 0]

if not forbidden_rows:
    print("この coefficients は Gb の禁止領域に一切触れていない（成功）。")
else:
    print(f"この coefficients は行 {forbidden_rows} の禁止領域に含まれている。")

# 同様にサイクルの制約も確認可能

cycle_check = []
for i, c in enumerate(constraints):
    c_prime = (Matrix([c]) * V).applyfunc(lambda x: x % P)
    val = (c_prime * coefficients)[0] % P
    if val == 0:
        cycle_check.append(i)

if not cycle_check:
    print("この coefficients はサイクルの制約にも触れていない。")
else:
    print(f"この coefficients はサイクル制約 {cycle_check} に抵触している。")

線形結合の係数ベクトル x:


Matrix([
[  3],
[709],
[689],
[746],
[762],
[732],
[ 68],
[ 84],
[565],
[557],
[744],
[346]])

検算成功: 一致しました。
Gb_prime * x (mod P) の結果:


Matrix([
[192],
[384]])

この coefficients は Gb の禁止領域に一切触れていない（成功）。
この coefficients はサイクルの制約にも触れていない。
